In [12]:
import pandas as pd

def clean_data(df):
    # Drop rows with missing data in column: 'review_text'
    df = df.dropna(subset=['review_text'])
    # Drop rows with missing data in column: 'likes'
    df = df.dropna(subset=['likes'])
    # Drop rows with missing data in column: 'comment_count'
    df = df.dropna(subset=['comment_count'])
    # Drop unnecessary columns (reviewer_name, reviewer_id, review_id)
    df = df.drop(columns=['reviewer_name', 'reviewer_id', 'review_id'])
    return df

# Loaded variable 'df' from URI: c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\combined_reviews_graphql.csv
df = pd.read_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\combined_reviews_graphql.csv')
df_clean = clean_data(df.copy())
df_clean.head()

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status
0,5,"To be fair, I was warned going in that this wa...",2019-01-21,1149.0,227.0,907,180,Elantris,Brandon Sanderson,False
1,3,"3.5/5 StarsExactly 3 months ago, I finished bi...",2017-03-16,474.0,99.0,4378,739,Elantris,Brandon Sanderson,False
2,4,"4.31!“Remember, the past need not become our f...",2022-04-30,348.0,132.0,3344,624,Elantris,Brandon Sanderson,False
3,4,Oh my God this is so contrary to my usual love...,2015-09-29,293.0,21.0,1987,356,Elantris,Brandon Sanderson,False
4,2,"Oh, Elantris, why must you torture me so? Why ...",2012-09-01,277.0,67.0,3450,563,Elantris,Brandon Sanderson,False


## Normalize reviews text 

In [13]:
import re

def clean_review(text):
    if not isinstance(text, str):
        return ""
    
    # lowercase
    text = text.lower()
    
    # normalize unicode quotes
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)
    
    # space out punctuation
    text = re.sub(r"([!?,.()])", r" \1 ", text)
    
    # remove control characters / strange symbols
    # but DO NOT remove emojis or punctuation
    text = re.sub(r"[^\w\s,.!?'\-—–…⭐️🤷‍♀️🤷‍♂️😍😡😢✨]+", " ", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df_clean["cleaned_review_text"] = df_clean["review_text"].apply(clean_review)


In [14]:
df_clean.head()

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status,cleaned_review_text
0,5,"To be fair, I was warned going in that this wa...",2019-01-21,1149.0,227.0,907,180,Elantris,Brandon Sanderson,False,"to be fair , i was warned going in that this w..."
1,3,"3.5/5 StarsExactly 3 months ago, I finished bi...",2017-03-16,474.0,99.0,4378,739,Elantris,Brandon Sanderson,False,"3 . 5 5 starsexactly 3 months ago , i finished..."
2,4,"4.31!“Remember, the past need not become our f...",2022-04-30,348.0,132.0,3344,624,Elantris,Brandon Sanderson,False,"4 . 31 ! remember , the past need not become o..."
3,4,Oh my God this is so contrary to my usual love...,2015-09-29,293.0,21.0,1987,356,Elantris,Brandon Sanderson,False,oh my god this is so contrary to my usual love...
4,2,"Oh, Elantris, why must you torture me so? Why ...",2012-09-01,277.0,67.0,3450,563,Elantris,Brandon Sanderson,False,"oh , elantris , why must you torture me so ? w..."


## Sentiment Analysis

### Multilingual Sentiment Classification Model -- 

- Hugging Face URL: https://huggingface.co/tabularisai/multilingual-sentiment-analysis

- I decided to use a multilingual model, since some of the reviews are in different languages.


In [17]:
pip install transformers

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   ----------- ---------------------------- 3.4/12.0 MB 12.2 MB/s eta 0:00:01
   --------------------------- ------------ 8.4/12.0 MB 21.0 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 20.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   ---------------------------------------- 566.1/566.1 kB 5.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 27.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
### Sentiment Analysis

# Import necessary libraries
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load the multilingual sentiment analysis model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("tabularisai/multilingual-sentiment-analysis")
model = AutoModelForSequenceClassification.from_pretrained("tabularisai/multilingual-sentiment-analysis")

# Create a sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True ## Some reviews may be longer than model's max length
)

# Example usage
sentiment_pipeline("I absolutely loved this book, it was amazing!")




Device set to use cpu


[{'label': 'Very Positive', 'score': 0.6916782259941101}]

In [24]:
## Run sentiment analysis on the cleaned reviews
# This will create a new column with the label
df_clean["sentiment_label"] = df_clean["cleaned_review_text"].apply(
    lambda x: sentiment_pipeline(x)[0]["label"]
)

# If you also want the confidence score
df_clean["sentiment_score"] = df_clean["cleaned_review_text"].apply(
    lambda x: sentiment_pipeline(x)[0]["score"]
)

In [25]:
df_clean.head()

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status,cleaned_review_text,sentiment_label,sentiment_score
0,5,"To be fair, I was warned going in that this wa...",2019-01-21,1149.0,227.0,907,180,Elantris,Brandon Sanderson,False,"to be fair , i was warned going in that this w...",Negative,0.562531
1,3,"3.5/5 StarsExactly 3 months ago, I finished bi...",2017-03-16,474.0,99.0,4378,739,Elantris,Brandon Sanderson,False,"3 . 5 5 starsexactly 3 months ago , i finished...",Negative,0.389702
2,4,"4.31!“Remember, the past need not become our f...",2022-04-30,348.0,132.0,3344,624,Elantris,Brandon Sanderson,False,"4 . 31 ! remember , the past need not become o...",Very Positive,0.835350
3,4,Oh my God this is so contrary to my usual love...,2015-09-29,293.0,21.0,1987,356,Elantris,Brandon Sanderson,False,oh my god this is so contrary to my usual love...,Very Negative,0.354678
4,2,"Oh, Elantris, why must you torture me so? Why ...",2012-09-01,277.0,67.0,3450,563,Elantris,Brandon Sanderson,False,"oh , elantris , why must you torture me so ? w...",Negative,0.797448


In [26]:
df_clean.to_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\cleaned_reviews_with_sentiment.csv', index=False)

In [27]:
df_clean.isnull().sum()

rating                 0
review_text            0
created_at             0
likes                  0
comment_count          0
review_length          0
word_count             0
book_title             0
book_author            0
spoiler_status         0
cleaned_review_text    0
sentiment_label        0
sentiment_score        0
dtype: int64

In [32]:
# Get the Average sentiment score and score per book
# Average score per book
avg_scores = df_clean.groupby("book_title")["sentiment_score"].mean().reset_index()
avg_scores.rename(columns={"sentiment_score": "avg_sentiment_score"}, inplace=True)

# Most common label per book
most_common_labels = (
    df_clean.groupby("book_title")["sentiment_label"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)
most_common_labels.rename(columns={"sentiment_label": "most_common_label"}, inplace=True)

# Combine both metrics
book_sentiment_summary = avg_scores.merge(most_common_labels, on="book_title")

book_sentiment_summary.head()

,book_title,avg_sentiment_score,most_common_label
0,Alcatraz Versus the Knights of Crystallia,0.699405,Positive
1,Alcatraz Versus the Scrivener's Bones,0.698265,Positive
2,Alcatraz Versus the Shattered Lens,0.690867,Positive
3,Arcanum Unbounded: The Cosmere Collection,0.672992,Positive
4,Bastille vs. the Evil Librarians,0.728929,Positive


In [30]:
df_details = pd.read_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\Combined_Details.csv')
df_details.head()

,Author,Title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...


In [37]:
df_details.rename(columns={"Title": "book_title"}, inplace=True)
df_details.head()

,Author,book_title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...


In [38]:
df_details = df_details.merge(
    book_sentiment_summary,
    on="book_title",
    how="left"
)
df_details.head()

,Author,book_title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL,avg_sentiment_score,most_common_label
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...,0.660050,Positive
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...,0.672504,Positive
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...,0.752283,Negative
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...,0.744905,Negative
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...,0.745153,Negative


In [39]:
df_details.to_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\book_details_with_sentiment.csv', index=False)

### Twitter Robertta Base Sentiment Analysis

In [44]:
#Load necessary libraries
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax


# Load the model
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)


# Example usage
text = "Covid cases are increasing fast!"

# Encode the text
encoded_input = tokenizer(text, return_tensors='pt')  # PyTorch

# Run the model
output = model(**encoded_input)

# Apply Softmax to get probabilities
scores = softmax(output[0][0].detach().numpy())

# Rank Labels
ranking = np.argsort(scores)[::-1]  # from highest to lowest probability
for i in range(scores.shape[0]):
    l = config.id2label[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i+1}) {l} {np.round(float(s), 4)}")




Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


1) negative 0.7236
2) neutral 0.2287
3) positive 0.0477


In [49]:
# Function to perform sentiment analysis on long texts
def twitter_roberta_sentiment_analysis(text, chunk_size=512):
    # No preprocessing needed for Goodreads reviews
    # text = preprocess(text)  

    # Tokenize into wordpieces
    tokens = tokenizer.tokenize(text)
    chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]
    
    all_scores = []

    for chunk in chunks:
        chunk_text = tokenizer.convert_tokens_to_string(chunk)
        encoded_input = tokenizer(chunk_text, return_tensors='pt', truncation=True, max_length=chunk_size)
        output = model(**encoded_input)
        scores = softmax(output[0][0].detach().numpy())
        all_scores.append(scores)

    avg_scores = np.mean(all_scores, axis=0)
    final_label = config.id2label[np.argmax(avg_scores)]
    final_score = np.max(avg_scores)
    
    return final_label, final_score


In [50]:
# Process sentiment analysis on cleaned reviews
df_clean[["sentiment_label_tR", "sentiment_score_tR"]] = df_clean["cleaned_review_text"].apply(
    lambda x: pd.Series(twitter_roberta_sentiment_analysis(x))
)



c:\Users\grega\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\grega\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [55]:
df_clean.head()

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status,cleaned_review_text,sentiment_label,sentiment_score,sentiment_label_tR,sentiment_score_tR
0,5,"To be fair, I was warned going in that this wa...",2019-01-21,1149.0,227.0,907,180,Elantris,Brandon Sanderson,False,"to be fair , i was warned going in that this w...",Negative,0.562531,negative,0.647602
1,3,"3.5/5 StarsExactly 3 months ago, I finished bi...",2017-03-16,474.0,99.0,4378,739,Elantris,Brandon Sanderson,False,"3 . 5 5 starsexactly 3 months ago , i finished...",Negative,0.389702,positive,0.593493
2,4,"4.31!“Remember, the past need not become our f...",2022-04-30,348.0,132.0,3344,624,Elantris,Brandon Sanderson,False,"4 . 31 ! remember , the past need not become o...",Very Positive,0.835350,positive,0.923146
3,4,Oh my God this is so contrary to my usual love...,2015-09-29,293.0,21.0,1987,356,Elantris,Brandon Sanderson,False,oh my god this is so contrary to my usual love...,Very Negative,0.354678,negative,0.618320
4,2,"Oh, Elantris, why must you torture me so? Why ...",2012-09-01,277.0,67.0,3450,563,Elantris,Brandon Sanderson,False,"oh , elantris , why must you torture me so ? w...",Negative,0.797448,neutral,0.449816


In [52]:
df_clean.to_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\cleaned_reviews_with_sentiment_V2.csv', index=False)

In [53]:
# Get the Average sentiment score and score per book
# Average score per book
avg_scores = df_clean.groupby("book_title")["sentiment_score_tR"].mean().reset_index()
avg_scores.rename(columns={"sentiment_score_tR": "avg_sentiment_score_tR"}, inplace=True)

# Most common label per book
most_common_labels = (
    df_clean.groupby("book_title")["sentiment_label_tR"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)
most_common_labels.rename(columns={"sentiment_label_tR": "most_common_label_tR"}, inplace=True)

# Combine both metrics
book_sentiment_summary = avg_scores.merge(most_common_labels, on="book_title")

book_sentiment_summary.head()

,book_title,avg_sentiment_score_tR,most_common_label_tR
0,Alcatraz Versus the Knights of Crystallia,0.842214,positive
1,Alcatraz Versus the Scrivener's Bones,0.849102,positive
2,Alcatraz Versus the Shattered Lens,0.814852,positive
3,Arcanum Unbounded: The Cosmere Collection,0.825217,positive
4,Bastille vs. the Evil Librarians,0.830221,positive


In [54]:
df_details = df_details.merge(
    book_sentiment_summary,
    on="book_title",
    how="left"
)
df_details.head()

,Author,book_title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL,avg_sentiment_score,most_common_label,avg_sentiment_score_tR,most_common_label_tR
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...,0.660050,Positive,0.790678,neutral
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...,0.672504,Positive,0.804084,positive
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...,0.752283,Negative,0.783775,positive
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...,0.744905,Negative,0.783771,negative
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...,0.745153,Negative,0.799931,positive


In [56]:
df_details.to_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\book_details_with_sentiment_V2.csv', index=False)

In [57]:
## One book didn't have any reviews processed (or at least no averages for sentiment score or label)

book_df = df_clean[df_clean["book_title"] == "Alcatraz Versus the Evil Librarians"].copy()
book_df

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status,cleaned_review_text,sentiment_label,sentiment_score,sentiment_label_tR,sentiment_score_tR


In [1]:
import pandas as pd
import numpy as np  

In [2]:
df_average = pd.read_csv(r'C:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\cleaned_reviews_with_sentiment_V2.csv')
df_average.head()

,rating,review_text,created_at,likes,comment_count,review_length,word_count,book_title,book_author,spoiler_status,cleaned_review_text,sentiment_label,sentiment_score,sentiment_label_tR,sentiment_score_tR
0,5,"To be fair, I was warned going in that this wa...",2019-01-21,1149.0,227.0,907,180,Elantris,Brandon Sanderson,False,"to be fair , i was warned going in that this w...",Negative,0.562531,negative,0.647602
1,3,"3.5/5 StarsExactly 3 months ago, I finished bi...",2017-03-16,474.0,99.0,4378,739,Elantris,Brandon Sanderson,False,"3 . 5 5 starsexactly 3 months ago , i finished...",Negative,0.389702,positive,0.593493
2,4,"4.31!“Remember, the past need not become our f...",2022-04-30,348.0,132.0,3344,624,Elantris,Brandon Sanderson,False,"4 . 31 ! remember , the past need not become o...",Very Positive,0.835350,positive,0.923146
3,4,Oh my God this is so contrary to my usual love...,2015-09-29,293.0,21.0,1987,356,Elantris,Brandon Sanderson,False,oh my god this is so contrary to my usual love...,Very Negative,0.354678,negative,0.618320
4,2,"Oh, Elantris, why must you torture me so? Why ...",2012-09-01,277.0,67.0,3450,563,Elantris,Brandon Sanderson,False,"oh , elantris , why must you torture me so ? w...",Negative,0.797448,neutral,0.449816


In [3]:
## Get averages of review_likes, comment_count, review_length, word_count

avg_metrics = df_average.groupby("book_title")[[
    "likes", "comment_count", "review_length", "word_count"
]].mean().reset_index()

avg_metrics.rename(columns={
    "likes": "avg_review_likes",
    "comment_count": "avg_comment_count",
    "review_length": "avg_review_length",
    "word_count": "avg_word_count"
}, inplace=True)

In [4]:
df_details = pd.read_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\book_details_with_sentiment_V2.csv')
df_details.head()

,Author,book_title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL,avg_sentiment_score,most_common_label,avg_sentiment_score_tR,most_common_label_tR
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...,0.660050,Positive,0.790678,neutral
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...,0.672504,Positive,0.804084,positive
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...,0.752283,Negative,0.783775,positive
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...,0.744905,Negative,0.783771,negative
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...,0.745153,Negative,0.799931,positive


In [5]:
df_details = df_details.merge(
    avg_metrics,
    on="book_title",
    how="left"
)


In [6]:
df_details.head()

,Author,book_title,Published Date,Audience Genre,Description,Average Rating,Total Ratings,Total Reviews,Non-Audience Genres,Currently Reading,Want to Read,URL,avg_sentiment_score,most_common_label,avg_sentiment_score_tR,most_common_label_tR,avg_review_likes,avg_comment_count,avg_review_length,avg_word_count
0,Brandon Sanderson,Elantris,"May 1, 2005",Adult,"Elantris was the capital of Arelon: gigantic, ...",4.17,340218,25290,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",13984,223889,https://www.goodreads.com/book/show/68427.Elan...,0.660050,Positive,0.790678,neutral,13.760695,1.856952,1230.415107,213.483289
1,Brandon Sanderson,Warbreaker,"June 9, 2009",Adult,From #1 New York Times bestselling author Bran...,4.30,271778,23677,"Fantasy, Fiction, Audiobook, High Fantasy, Epi...",12231,153250,https://www.goodreads.com/book/show/1268479.Wa...,0.672504,Positive,0.804084,positive,14.348059,1.884873,1176.048862,205.964525
2,Brandon Sanderson,"White Sand, Volume 1","June 21, 2016",Not specified,A brand new saga of magic and adventure by #1 ...,3.59,24022,2183,"Fantasy, Graphic Novels, Comics, Fiction, High...",2004,19947,https://www.goodreads.com/book/show/28862254-w...,0.752283,Negative,0.783775,positive,1.393858,0.139519,552.549399,98.321762
3,Brandon Sanderson,"White Sand, Volume 2","February 21, 2018",Adult,Following the loss of most of his colleagues i...,3.53,14097,995,"Fantasy, Graphic Novels, Comics, Fiction, High...",567,12635,https://www.goodreads.com/book/show/33551363-w...,0.744905,Negative,0.783771,negative,0.935549,0.165156,386.010070,69.239678
4,Brandon Sanderson,"White Sand, Volume 3","September 18, 2019",Not specified,"Underpowered and overwhelmed, Kenton tries to ...",3.64,11587,859,"Fantasy, Graphic Novels, Comics, Fiction, High...",321,10233,https://www.goodreads.com/book/show/39298848-w...,0.745153,Negative,0.799931,positive,0.838184,0.080326,426.216531,76.128056


In [7]:
df_details.to_csv(r'c:\Programming\Python\msba_capstone_dragonsteel\Goodreads Data\book_details_with_sentiment_V2_averages.csv', index=False)